<a href="https://colab.research.google.com/github/Allah-Bakhsh/flyrank-ml-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Allah-Bakhsh/flyrank-ml-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal checks first (below):** staleness (content age vs. decline rate) and CTR-vs-position (behind FlyRank's CTR-fix logic).

**The rule:** for each page, compare its actual CTR to the *expected* CTR for its position bucket (measured directly from this month's data, Signal 2 below). If a page's CTR falls meaningfully below what its position bucket would predict, and it gets meaningful impression volume, it's flagged for review weighted by impressions, since a CTR gap on a high-traffic page matters more than the same gap on a page nobody sees.

**Reason codes:** `weak_ctr_for_position` (flagged) or `ctr_at_or_above_expected` (not flagged).
**Action labels:** `review_for_refresh` or `no_action_needed`.

In [2]:
# --- Setup ---
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

import duckdb
import pandas as pd

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

MARCH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

# --- Signal 1: staleness, behind FlyRank's refresh flags ---
!wget -q https://raw.githubusercontent.com/Allah-Bakhsh/flyrank-ml-internship-week1/main/data/raw/content_refresh_anonymized.csv -O content_refresh_anonymized.csv
starter = pd.read_csv("content_refresh_anonymized.csv")

starter["age_bucket"] = pd.cut(
    starter["content_age_days"],
    bins=[0, 90, 180, 365, 10_000],
    labels=["0-90d", "91-180d", "181-365d", "365d+"]
)

signal1 = starter.groupby("age_bucket", observed=True).agg(
    n=("content_age_days", "size"),
    pct_declining=("trend_direction", lambda s: (s == "down").mean())
)
print("Signal 1 — staleness vs. decline rate (starter dataset):")
display(signal1)
print("Verdict: OPPOSITE — older pages are not more likely to be declining here; decline rate doesn't rise with age. Staleness alone is not used as a rule input.\n")

# --- Signal 2: CTR vs. position, behind FlyRank's CTR-fix logic ---
ctr_vs_position = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3 THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            WHEN gsc_avg_position <= 50 THEN '21-50'
            ELSE '50+'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) AS avg_ctr
    FROM '{MARCH}'
    WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    GROUP BY 1
    ORDER BY MIN(gsc_avg_position)
""").df()
print("Signal 2 — CTR by position bucket (warehouse, March 2026):")
display(ctr_vs_position)
print("Verdict: CONFIRMED — CTR drops as position worsens, matching FlyRank's CTR-fix logic. This signal drives the rule below.")


Signal 1 — staleness vs. decline rate (starter dataset):


,n,pct_declining
age_bucket,,
0-90d,492,0.668699
91-180d,11780,0.625552
181-365d,11368,0.514866
365d+,6360,0.426258


Verdict: OPPOSITE — older pages are not more likely to be declining here; decline rate doesn't rise with age. Staleness alone is not used as a rule input.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Signal 2 — CTR by position bucket (warehouse, March 2026):


,position_bucket,n,avg_ctr
0,1-3,727362,0.004756
1,4-10,1456122,0.003473
2,11-20,519223,0.002770
3,21-50,631491,0.001638
4,50+,276863,0.000494


Verdict: CONFIRMED — CTR drops as position worsens, matching FlyRank's CTR-fix logic. This signal drives the rule below.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Uses Signal 2 (CONFIRMED) as the benchmark; Signal 1 was ruled out, so staleness isn't part of the score.

In [3]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        COUNT(*) AS days_observed,
        AVG(gsc_impressions) AS avg_impressions,
        AVG(gsc_clicks) AS avg_clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr
    FROM '{MARCH}'
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

def position_bucket(pos):
    if pos <= 3: return "1-3"
    elif pos <= 10: return "4-10"
    elif pos <= 20: return "11-20"
    elif pos <= 50: return "21-50"
    else: return "50+"

features["position_bucket"] = features["avg_position"].apply(position_bucket)
expected_ctr = ctr_vs_position.set_index("position_bucket")["avg_ctr"]
features["expected_ctr"] = features["position_bucket"].map(expected_ctr)

features["ctr_gap"] = features["expected_ctr"] - features["ctr"]
features["score"] = features["ctr_gap"].clip(lower=0) * features["avg_impressions"]

features["reason_code"] = "weak_ctr_for_position"
features["action"] = "review_for_refresh"
features.loc[features["ctr_gap"] <= 0, "action"] = "no_action_needed"
features.loc[features["ctr_gap"] <= 0, "reason_code"] = "ctr_at_or_above_expected"

queue = features.sort_values("score", ascending=False).reset_index(drop=True)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
display(queue.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 176738 rows to work/outputs/baseline_action_score.csv


,content_hash_id,days_observed,avg_impressions,avg_clicks,avg_position,ctr,position_bucket,expected_ctr,ctr_gap,score,reason_code,action
0,content_8d7d99f109e19aa2,29,7017.137931,9.965517,2.563756,0.001420,1-3,0.004756,0.003335,23.404592,weak_ctr_for_position,review_for_refresh
1,content_44f34c0a90047651,31,6851.741935,0.774194,7.346909,0.000113,4-10,0.003473,0.003360,23.019409,weak_ctr_for_position,review_for_refresh
2,content_8e1334d6356668e3,31,4354.322581,0.032258,4.545582,0.000007,4-10,0.003473,0.003465,15.088717,weak_ctr_for_position,review_for_refresh
3,content_34a70fea29d15f24,31,4613.516129,1.387097,3.219473,0.000301,4-10,0.003473,0.003172,14.633963,weak_ctr_for_position,review_for_refresh
4,content_fec55986a1868d62,31,4002.419355,0.032258,9.385150,0.000008,4-10,0.003473,0.003465,13.866686,weak_ctr_for_position,review_for_refresh
5,content_7c6373141eae744a,31,4277.193548,2.677419,5.789019,0.000626,4-10,0.003473,0.002847,12.175715,weak_ctr_for_position,review_for_refresh
6,content_306bc78dff1eb683,29,2786.931034,1.206897,1.488604,0.000433,1-3,0.004756,0.004322,12.046398,weak_ctr_for_position,review_for_refresh
7,content_f6116743b00afc2d,31,3470.451613,0.483871,9.536301,0.000139,4-10,0.003473,0.003333,11.567743,weak_ctr_for_position,review_for_refresh
8,content_0e03de7680314cd5,29,7631.379310,24.827586,2.675217,0.003253,1-3,0.004756,0.001502,11.463558,weak_ctr_for_position,review_for_refresh
9,content_9ef3d7516483e665,29,3076.862069,3.172414,2.481596,0.001031,1-3,0.004756,0.003724,11.459652,weak_ctr_for_position,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = queue.head(20).reset_index(drop=True)
for i, row in top20.iterrows():
    print(f"{i+1}. content={row['content_hash_id']}  score={row['score']:.1f}  action={row['action']}  reason={row['reason_code']}")
    print(f"   Why: position bucket {row['position_bucket']} expects CTR ~{row['expected_ctr']:.3f}, this page gets {row['ctr']:.3f}, on {row['avg_impressions']:.0f} avg impressions/day.")
    print(f"   Would be wrong if: the low CTR is from mismatched search intent, a SERP feature change, or seasonal dip — not actually weak content. A reviewer should check the live page before acting.\n")

1. content=content_8d7d99f109e19aa2  score=23.4  action=review_for_refresh  reason=weak_ctr_for_position
   Why: position bucket 1-3 expects CTR ~0.005, this page gets 0.001, on 7017 avg impressions/day.
   Would be wrong if: the low CTR is from mismatched search intent, a SERP feature change, or seasonal dip — not actually weak content. A reviewer should check the live page before acting.

2. content=content_44f34c0a90047651  score=23.0  action=review_for_refresh  reason=weak_ctr_for_position
   Why: position bucket 4-10 expects CTR ~0.003, this page gets 0.000, on 6852 avg impressions/day.
   Would be wrong if: the low CTR is from mismatched search intent, a SERP feature change, or seasonal dip — not actually weak content. A reviewer should check the live page before acting.

3. content=content_8e1334d6356668e3  score=15.1  action=review_for_refresh  reason=weak_ctr_for_position
   Why: position bucket 4-10 expects CTR ~0.003, this page gets 0.000, on 4354 avg impressions/day.
   Wou

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
weak_picks = top20[top20["avg_impressions"] < 5]
print("Weak picks (low impression volume — score may be noisy from small n):")
display(weak_picks[["content_hash_id", "avg_impressions", "score"]])

print("""
Leakage check:
- Only March 2026 data used — no April-June, no future window touched.
- No FlyRank product flags (health_score, priority_score, decision flags) were loaded or used as inputs.
- expected_ctr benchmark was computed from this same March slice, acceptable for a baseline rule, but would need a held-out benchmark for real model evaluation later.
""")

Weak picks (low impression volume — score may be noisy from small n):


,content_hash_id,avg_impressions,score



Leakage check:
- Only March 2026 data used — no April-June, no future window touched.
- No FlyRank product flags (health_score, priority_score, decision flags) were loaded or used as inputs.
- expected_ctr benchmark was computed from this same March slice, acceptable for a baseline rule, but would need a held-out benchmark for real model evaluation later.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.